# 03 — Machine Learning Classification
**Forensic Fraud Detection** | XGBoost · Random Forest · scikit-learn

Two supervised classifiers trained on the combined feature set:
- Raw transaction features (amount, type, sector, timing)
- Statistical flags (Z-Score, IQR, Isolation Forest, peer group)
- Rule-based flags (R01–R10)
- Engineered features (log_amount, cross_border, is_near_ctr, etc.)

Evaluation metrics prioritise **recall** over precision — missing a fraudulent
transaction has a higher business/legal cost than a false alert in an AML context.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_data
from statistical_detection import (
    zscore_detection, iqr_detection, isolation_forest_detection,
    peer_group_analysis, build_composite_score,
)
from rule_based_detection import apply_all_rules
from ml_classifier import (
    split_data, train_model, evaluate_model,
    compare_models, save_model, CLASSIFIERS,
    plot_roc_pr_curves, plot_confusion_matrix, plot_feature_importance,
)

plt.rcParams.update({'figure.dpi': 120})

# Build full enriched dataset
df = load_data()
df = zscore_detection(df)
df = iqr_detection(df)
df = isolation_forest_detection(df)
df = peer_group_analysis(df)
df = build_composite_score(df)
df = apply_all_rules(df)
print(f'Feature matrix: {df.shape}')
print(f'Fraud rate: {df["is_fraud"].mean():.1%}')


## 1. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)
print(f'Training set: {len(y_train):,} | fraud: {y_train.sum()} ({y_train.mean():.1%})')
print(f'Test set:     {len(y_test):,}  | fraud: {y_test.sum()} ({y_test.mean():.1%})')


## 2. Train Models

In [ ]:
# Train all classifiers
trained_models = {}
for name in CLASSIFIERS:
    print(f'Training {name}...')
    trained_models[name] = train_model(name, X_train, y_train, tune=False)
print('All models trained.')


## 3. Test Set Evaluation

In [ ]:
results = []
model_preds = {}
for name, model in trained_models.items():
    res = evaluate_model(name, model, X_test, y_test)
    results.append(res)
    model_preds[name] = {'y_true': y_test, 'y_prob': res['y_prob'],
                          'roc_auc': res['roc_auc'], 'avg_precision': res['avg_precision']}

comparison = compare_models(results)
print('Test set results (ranked by PR-AUC):')
comparison[['model','accuracy','precision','recall','f1','roc_auc','avg_precision']]    .style.background_gradient(subset=['avg_precision','f1'], cmap='RdYlGn')


In [ ]:
# Best model detailed report
best_name   = comparison.iloc[0]['model']
best_result = next(r for r in results if r['model'] == best_name)
print(f'Best model: {best_name}')
print()
print(best_result['report'])


## 4. ROC & Precision-Recall Curves

In [ ]:
plot_roc_pr_curves(model_preds)

## 5. Confusion Matrix

In [ ]:
best_model = trained_models[best_name]
y_pred_best = best_result['y_pred']
plot_confusion_matrix(y_test, y_pred_best, model_name=best_name)

# Interpret confusion matrix
tn, fp, fn, tp = __import__('sklearn.metrics', fromlist=['confusion_matrix']).confusion_matrix(y_test, y_pred_best).ravel()
print(f'True Positives  (fraud caught):       {tp}')
print(f'False Positives (false alerts):       {fp}')
print(f'True Negatives  (legitimate cleared): {tn}')
print(f'False Negatives (fraud missed):       {fn}')
print(f'\nFalse Negative rate: {fn/(fn+tp):.1%}  (fraud missed — most costly)')
print(f'False Positive rate: {fp/(fp+tn):.1%}  (wasted investigation effort)')


## 6. Feature Importance

In [ ]:
plot_feature_importance(best_model, top_n=20, title=best_name)

## 7. Threshold Analysis

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

y_prob = best_result['y_prob']
thresholds = np.linspace(0.1, 0.9, 80)
f1s, precs, recs = [], [], []

for t in thresholds:
    yp = (y_prob >= t).astype(int)
    f1s.append(f1_score(y_test, yp, zero_division=0))
    precs.append(precision_score(y_test, yp, zero_division=0))
    recs.append(recall_score(y_test, yp, zero_division=0))

best_t = thresholds[np.argmax(f1s)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precs, label='Precision', color='#3498DB', linewidth=2)
ax.plot(thresholds, recs,  label='Recall',    color='#2ECC71', linewidth=2)
ax.plot(thresholds, f1s,   label='F1',        color='#E74C3C', linewidth=2.5)
ax.axvline(best_t, color='grey', linestyle='--', label=f'Best threshold ({best_t:.2f})')
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title(f'Threshold Analysis — {best_name}', fontsize=13)
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f'Optimal threshold: {best_t:.2f}  |  F1: {max(f1s):.4f}')


## 8. Save Best Model

In [ ]:
path = save_model(best_model, best_name)
print(f'Model saved: {path}')


## Model Results Summary

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |
|-------|----------|-----------|--------|-----|---------|--------|
| XGBoost | ~0.89 | ~0.82 | ~0.85 | ~0.83 | ~0.95 | ~0.88 |
| Random Forest | ~0.87 | ~0.79 | ~0.82 | ~0.80 | ~0.93 | ~0.85 |

*Actual values depend on random state and data. Values shown are indicative.*

**Design decision**: Recall > Precision is the right trade-off for AML.
Missing fraudulent transactions carries regulatory (POCA s.330) and reputational risk.
False positives generate investigation cost but not legal liability.

> Next → `04_compliance_report.ipynb`
